# 02 · Prefetch stages: more is not better

Prefetching K-tiles ahead of the mainloop hides DRAM latency. More prefetch should hide more latency. That reasoning is intuitive, widely believed, and **wrong past a point** — and the trace shows precisely where it breaks and why.

## 1. The question

`prefetch_stages` controls how many K-tile loads are issued before the mainloop begins. We sweep three values on identical hardware and shape:

- **PS0** — no prefetch
- **PS2** — two prefetch iterations
- **PS3** — three prefetch iterations

**Hypothesis:** PS3 hides more latency than PS2 and is therefore faster.

Hold on to that prediction. It is wrong, and the interesting part is *which range* reveals it.

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        595.58.03
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Identical in all three variants — only `prefetch_stages` differs. Two ranges carry the whole argument:

| Range | Meaning | Who waits |
|---|---|---|
| `cons_wait` | MMA waiting for data to arrive | **consumer** starved |
| `prod_acquire` | TMA waiting for a free smem stage | **producer** back-pressured |


These are the two ends of the same pipeline. If the pipeline is badly fed, `cons_wait` grows. If it is over-fed, there is no free stage to load into and `prod_acquire` grows. **The bubble does not vanish; it moves.** Watch which one.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_0x_iket_ps_2_detailed.py"),
    limit=12,
)

fp16_gemm_0x_iket_ps_2_detailed.py: 32 IKET call sites (showing first 12)
------------------------------------------------------------------------
>   86      cute.experimental.iket.range_push("prologue")
       ...
>  227      cute.experimental.iket.range_pop()
       ...
>  232      cute.experimental.iket.range_push("mainloop")
       ...
>  238          cute.experimental.iket.range_push("prefetch")
       ...
>  240              cute.experimental.iket.range_push("prefetch_iter")
       ...
>  242              cute.experimental.iket.range_push("prod_acquire")
       ...
>  244              cute.experimental.iket.range_pop()
       ...
>  247              cute.experimental.iket.range_push("tma_issue")
       ...
>  260              cute.experimental.iket.range_pop()
>  261              cute.experimental.iket.range_pop()
>  262          cute.experimental.iket.range_pop()
       ...
>  265              cute.experimental.iket.range_push("k_tile_iter")


## 3. Run it

Three runs, sequentially. Running them concurrently would let the kernels perturb each other's timings and invalidate the comparison.

In [2]:
for tag, src in [("ps0", "fp16_gemm_0x_iket_ps_0_detailed.py"),
                 ("ps2", "fp16_gemm_0x_iket_ps_2_detailed.py"),
                 ("ps3", "fp16_gemm_0x_iket_ps_3_detailed.py")]:
    cmd = (
        f"run-iket --output-dir out/nb02_{tag} --clobber profile --postprocess all -- "
        f"python {nbhelp.kernel('instrumented', src)} --mnk 4096,4096,512"
    )
    print(f"===== {tag} =====")
    check_env.run_or_show(cmd, env, est="about 40 s each")

===== ps0 =====
$ run-iket --output-dir out/nb02_ps0 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============
Running Blackwell fp16 GEMM example 0 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

a: tensor([[ 1., -2., -2.,  ..., -2., -1.,  1.],
        [ 0.,  1., -1.,  ...,  0., -2.,  0.],
        [ 0., -1.,  1.,  ..., -1.,  0.,  0.],
        ...,
        [ 0., -2.,  0.,  ...,  1., -2.,  0.],
        [-1., -1., -1.,  ..., -1.,  0.,  1.],
        [ 1.,  0., -2.,  ...,  0., -1.,  0.]], device='cuda:0',
       dtype=torch.float16)
b: tensor([[-2., -2.,  0.,  ...,  1., -1., -2.],
        [ 1., -2., -1.,  ...,  0., -2.,  0.],
        [-1., -1., -2.,  ...,  0., -1., -1.],
        ...,
        [ 1.,  0.,  0.,  ..., -1., -1., -1.],
        [-2., -2.,  1.,  ...,  0.,  1., -2.],
        [ 1.,  1.,  0.,  ...,  1.,  0., -1.]], device='cuda:0',
       dt

## 4. The trace

**PS0 — an unfed pipeline.** Every `k_tile_iter` carries a long `cons_wait`: the MMA warp blocks on data for every single tile, because no load was issued ahead.

In [4]:
nbhelp.figure("nb01/gemm0x_ps0_baseline_trace.png", "PS0: a long cons_wait in every iteration. The prefetch block exists but runs zero iterations.")

**PS2 — smooth.** The prefetch block now runs before the mainloop and keeps stages full. `cons_wait` collapses; loads and issues interleave with no visible stall.

In [5]:
nbhelp.figure("nb02/gemm0x_ps2_smooth_pipeline.png", "PS2: cons_wait has collapsed. This is what a fed pipeline looks like.")

**PS3 — the bubble returns, somewhere else.** `cons_wait` is now *even smaller* than PS2. But the `prod_acquire` bands have visibly widened.

In [6]:
nbhelp.figure("nb02/gemm0x_ps3_prod_acquire_bubble.png", "PS3: prod_acquire widens. All stages are full, so the TMA warp waits for one to free.")

In [3]:
import json, pathlib
for tag in ("ps0", "ps2", "ps3"):
    d = pathlib.Path(f"out/nb02_{tag}")
    if not d.exists():
        print(f"{tag}: not collected locally")
        continue
    s = ts.summarize(ts.load(ts.find_trace(d)))
    m = {r.name: r.median_us for r in s.ranges}
    print(f"{tag}: span={s.span_us:7.2f}us  cons_wait={m.get('cons_wait', float('nan')):.3f}us  "
          f"prod_acquire={m.get('prod_acquire', float('nan')):.3f}us")

ps0: span=  39.26us  cons_wait=0.544us  prod_acquire=0.064us
ps2: span=  26.08us  cons_wait=0.064us  prod_acquire=0.032us
ps3: span=  28.80us  cons_wait=0.032us  prod_acquire=0.128us


## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512`:

| | PS0 | PS2 | PS3 |
|---|---|---|---|
| Warp-lifetime span | 39.6 µs | **26.2 µs** | 28.9 µs |
| `cons_wait` median | **0.544 µs** | 0.032-0.064 µs | 0.032 µs |
| `prod_acquire` median | 0.064 µs | 0.064 µs | **0.128 µs** |
| `k_tile_iter` median | 0.768 µs | 0.288 µs | 0.416 µs |
| `prefetch` median | 0.000 µs | 0.224 µs | 0.320 µs |


**PS0 → PS2.** `cons_wait` falls at least 8.5×, from 0.544 µs down to one or two timestamp quanta. The span drops 34%. Prefetch is doing exactly what it promised. Note `prefetch` at PS0 measures 0.000 µs — the block is present but runs zero iterations, which is how the trace tells you the knob is off.

**PS2 → PS3, the interesting one.** `cons_wait` does not distinguish them: both sit at 0.032–0.064 µs, i.e. at the resolution floor, and the exact value moves between runs. By that metric PS3 looks fine, or even better.

But the span gets **worse**, 26.2 → 28.9 µs, about 10% slower. The discriminating range is the *other* one: `prod_acquire` **doubles**, 0.064 → 0.128 µs, consistently across runs. With three stages pre-filled there is no free smem stage for the TMA warp to load into, so the producer stalls.

The bubble moved from the consumer to the producer. A profile that only watched `cons_wait` would have concluded PS3 was the better configuration.

**The transferable rule:** more prefetch is not automatically better. Instrument *both ends* of a pipeline, because relieving one end can load the other.

**Next:** [03 · 2-CTA MMA and multicast](03_2cta_multicast.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*